# Table context extraction hybrid (Unlimited-OCR + GLiNER + Ollama)

Same JSON structure and pipeline as `extract_table_context_ollama_lighonocr.ipynb`,
but OCR is [baidu/Unlimited-OCR](https://huggingface.co/baidu/Unlimited-OCR)
([HF Space demo](https://huggingface.co/spaces/baidu/Unlimited-OCR)).

Backend split:
- OCR: Unlimited-OCR via HF Space API (default), or optional SGLang / local CUDA
- `datasets` and `metrics`: GLiNER2
- `mentions` context refinement: Ollama (`qwen3:1.7b`)

Output schema:
- `paper`, `source_pdf`, `num_tables`
- `tables[]` with `table_id`, `table_label`, `page`, `caption`, `mentions`, `datasets`, `metrics`

**Setup notes**
- **Just run the notebook**: default backend is `OCR_BACKEND="hf_space"` (calls the public [HF Space](https://huggingface.co/spaces/baidu/Unlimited-OCR) via API — no SGLang, no local CUDA for OCR).
- Needs **internet**. Set `HF_TOKEN` (Hugging Face access token) for more ZeroGPU quota on the public Space.
- Alternatives: `OCR_BACKEND="sglang"` (local server) or `"local"` (CUDA + exact torch 2.10.0).
- PDF OCR uses **base mode** per the official README.
- OCR cache and JSON outputs go to `pdfs_test/out_unlimited/`.

In [ ]:
# Runtime deps — run once per server, then restart kernel if anything was installed.
%pip install -q gradio-client

# Only needed if OCR_BACKEND="local"
%pip install -q addict==2.4.0 easydict==1.13

## Optional: SGLang setup (only if `OCR_BACKEND = "sglang"`)

Skip this section if you use the default `hf_space` backend — the notebook calls the HF Space API directly.

```bash
python -m sglang.launch_server \
    --model baidu/Unlimited-OCR \
    --served-model-name Unlimited-OCR \
    --attention-backend fa3 --page-size 1 \
    --mem-fraction-static 0.8 --context-length 32768 \
    --enable-custom-logit-processor --disable-overlap-schedule \
    --skip-server-warmup --host 0.0.0.0 --port 10000
```

In [ ]:
from __future__ import annotations

import json
import logging
import os
import re
import base64
import tempfile
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import fitz
import requests
import torch
from bs4 import BeautifulSoup
from PIL import Image

In [ ]:
# Paths and run options (same behavior as script)
INPUT_PATH = Path("pdfs_test")
OUTPUT_DIR = Path("pdfs_test/out_unlimited")
OCR_CACHE_DIR = OUTPUT_DIR / "ocr_cache"

FORCE_OCR = False
SKIP_EXISTING = False

# Hybrid mode
# - Entities (datasets/metrics): GLiNER2
# - Context mentions: Ollama
USE_OLLAMA_FOR_CONTEXT = True

# Ollama
OLLAMA_MODEL = "qwen3:1.7b"
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_TIMEOUT = 120

# GLiNER2
GLINER2_MODEL_ID = "fastino/gliner2-base-v1"
GLINER2_MIN_SCORE = 0.65
GLINER2_MAX_CHARS = 3000
GLINER2_FORCE_CPU = True  # Set False if CUDA+nvrtc stack is healthy

# Unlimited-OCR
# - "hf_space": HF Space API via gradio_client (default — notebook only, needs internet)
# - "sglang": OpenAI-compatible local server
# - "local": HuggingFace Transformers on CUDA (fragile unless torch==2.10.0)
OCR_BACKEND = "hf_space"
HF_SPACE_ID = "baidu/Unlimited-OCR"
HF_SPACE_URL = "https://baidu-unlimited-ocr.hf.space"
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
HF_SPACE_TIMEOUT = 900  # seconds per OCR request
HF_SPACE_RETRIES = 3
HF_SPACE_PDF_MODE = "gundam"
OCR_MODEL_ID = "baidu/Unlimited-OCR"
OCR_RENDER_DPI = 200
OCR_MAX_PAGE_LONGEST = 1024  # align with PDF base mode
OCR_TORCH_DTYPE = "bfloat16"  # local infer() hardcodes cuda autocast to bfloat16
OCR_PROMPT = "<image>document parsing."
OCR_MAX_LENGTH = 32768
OCR_NO_REPEAT_NGRAM_SIZE = 35
OCR_NGRAM_WINDOW = 128
OCR_BASE_SIZE = 1024
OCR_IMAGE_SIZE = 640
OCR_CROP_MODE = True  # gundam mode (single-image only; not used for PDFs)

# PDF OCR — official README uses base mode (image_size=1024, crop_mode=False)
OCR_PDF_PROMPT = "<image>Multi page parsing."
OCR_PDF_IMAGE_SIZE = 1024
OCR_PDF_BASE_SIZE = 1024
OCR_PDF_CROP_MODE = False
OCR_PDF_NGRAM_WINDOW = 1024
OCR_PDF_STRATEGY = "per_page_base"

SGLANG_SERVER_URL = "http://127.0.0.1:10000"
SGLANG_MODEL_NAME = "Unlimited-OCR"

In [ ]:
# Standalone pipeline (Unlimited-OCR + GLiNER entities + Ollama context)

ADJACENT_PARA_WINDOW = 2
MIN_MENTION_CHARS = 40

_HEADING_ONLY_RE = re.compile(r"^#{1,6}\s+\S")
_FOOTNOTE_URL_RE = re.compile(r"https?://|github\.com|\$\^\{\d+\}")

_NUM = r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+"
TABLE_BLOCK_RE = re.compile(r"<table\b[^>]*>.*?</table>", re.DOTALL | re.IGNORECASE)
CAPTION_RE = re.compile(
    rf"(?:\*\*)?(?:Table|Tab\.?|TABLE)\s+({_NUM})(?:\*\*)?\s*[:.\u2014-]\s+",
    re.IGNORECASE,
)
TABLE_REF_RE = re.compile(
    rf"\b(?:Table|Tab\.?|TABLE|Tables|TABLES)\s+({_NUM})(?:\s*(?:,|and|&)\s*({_NUM}))*",
    re.IGNORECASE,
)
_REF_KEYWORD_RE = re.compile(r"^(?:Tables?|Tab\.?|TABLES?)\s*", re.IGNORECASE)
_REF_NUM_RE = re.compile(r"[A-Za-z]?\d+(?:\.\d+)?|[IVXLC]+")

_latex2text = None
_LATEX_MATH_RE = re.compile(r"\$\$([^$]+)\$\$|\$([^$]+)\$|\\\(([^)]+)\\\)|\\\[([^\]]+)\\\]")

_HYPHEN_CONTINUATION_PREFIX_RE = re.compile(r"^([a-z][a-z'-]{0,30})([.,;:!?])?")
_SENTENCE_END_RE = re.compile(r"""[.!?]['")\]]*\s*$""")
MAX_MENTION_EXTENSIONS = 2

GLINER2_LABEL_DESCRIPTIONS = {
    "model": (
        "Name of a model, method, or algorithm "
        "(e.g. TransE, ComplEx, RotatE)."
    ),
    "dataset": (
        "Name of a benchmark dataset or knowledge-graph corpus "
        "(e.g. WN18, FB15k, YAGO)."
    ),
    "metric": (
        "Name of an evaluation metric used for reporting performance "
        "(e.g. MRR, Hits@10, F1)."
    ),
}
GLINER2_LABELS = list(GLINER2_LABEL_DESCRIPTIONS.keys())

log = logging.getLogger("extract_table_context_unlimitedocr")

ocr_tokenizer = None
ocr_model = None
hf_space_client = None
gliner2_model = None

_UNLIMITED_DET_RE = re.compile(r"<\|det\|>.*?<\|/det\|>", re.DOTALL)
_UNLIMITED_REF_RE = re.compile(r"<\|ref\|>(.*?)<\|/ref\|>", re.DOTALL)
_MD_TABLE_ROW_RE = re.compile(r"^\s*\|.+\|\s*$")
_MD_TABLE_SEP_RE = re.compile(r"^\s*\|?(?:\s*:?-{3,}:?\s*\|)+\s*$")


def _configure_torch_for_ocr() -> None:
    os.environ.setdefault("PYTORCH_JIT", "0")
    os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")
    if not torch.cuda.is_available():
        return
    torch.backends.cuda.enable_cudnn_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cudnn.benchmark = False
    _patch_sdpa_for_unlimited_ocr()


def _resolve_ocr_dtype():
    # Unlimited-OCR infer() always wraps generation in torch.autocast("cuda", dtype=bfloat16).
    # Loading with another dtype causes conv/linear type mismatches.
    if OCR_BACKEND == "local":
        requested = str(OCR_TORCH_DTYPE).lower()
        if requested not in {"bfloat16", "bf16"}:
            print(
                f"Warning: OCR_TORCH_DTYPE={OCR_TORCH_DTYPE!r} ignored for local backend; "
                "using bfloat16 to match model.infer()."
            )
        return torch.bfloat16
    name = str(OCR_TORCH_DTYPE).lower()
    if name == "float32":
        return torch.float32
    if name == "float16":
        return torch.float16
    return torch.bfloat16


def _prepare_ocr_model(model) -> None:
    for module in model.modules():
        if hasattr(module, "use_flash_attention"):
            module.use_flash_attention = False


def _manual_scaled_dot_product_attention(
    query,
    key,
    value,
    attn_mask=None,
    dropout_p: float = 0.0,
    is_causal: bool = False,
    scale=None,
    **kwargs,
):
    scale_factor = scale if scale is not None else query.size(-1) ** -0.5
    attn_weight = torch.matmul(query, key.transpose(-2, -1)) * scale_factor
    if is_causal:
        seq_len = query.size(-2)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=query.device, dtype=torch.bool),
            diagonal=1,
        )
        attn_weight = attn_weight.masked_fill(causal_mask, float("-inf"))
    if attn_mask is not None:
        attn_weight = attn_weight + attn_mask.to(dtype=attn_weight.dtype, device=attn_weight.device)
    attn_weight = torch.softmax(attn_weight, dim=-1, dtype=torch.float32)
    attn_weight = torch.nan_to_num(attn_weight, nan=0.0).to(query.dtype)
    if dropout_p > 0.0:
        attn_weight = torch.nn.functional.dropout(attn_weight, p=dropout_p)
    return torch.matmul(attn_weight, value)


_SDPA_ORIG = None
_SDPA_PATCHED = False


def _patch_sdpa_for_unlimited_ocr() -> None:
    """Unlimited-OCR SAM encoder uses attn_mask SDPA, which can crash in cuDNN."""
    global _SDPA_ORIG, _SDPA_PATCHED
    if _SDPA_PATCHED:
        return
    import torch.nn.functional as F

    _SDPA_ORIG = F.scaled_dot_product_attention

    def _safe_scaled_dot_product_attention(*args, **kwargs):
        try:
            return _SDPA_ORIG(*args, **kwargs)
        except RuntimeError as exc:
            msg = str(exc).lower()
            if "cudnn" in msg or "cuda" in msg or "cublas" in msg:
                return _manual_scaled_dot_product_attention(*args, **kwargs)
            raise

    F.scaled_dot_product_attention = _safe_scaled_dot_product_attention
    _SDPA_PATCHED = True


def _run_unlimited_ocr_infer(image_path: str, out_dir: str, **overrides) -> str:
    settings = {
        "prompt": OCR_PROMPT,
        "base_size": OCR_BASE_SIZE,
        "image_size": OCR_IMAGE_SIZE,
        "crop_mode": OCR_CROP_MODE,
        "ngram_window": OCR_NGRAM_WINDOW,
    }
    settings.update(overrides)
    _configure_torch_for_ocr()
    with torch.inference_mode():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        result = ocr_model.infer(
            ocr_tokenizer,
            prompt=settings["prompt"],
            image_file=image_path,
            output_path=out_dir,
            base_size=settings["base_size"],
            image_size=settings["image_size"],
            crop_mode=settings["crop_mode"],
            max_length=OCR_MAX_LENGTH,
            no_repeat_ngram_size=OCR_NO_REPEAT_NGRAM_SIZE,
            ngram_window=settings["ngram_window"],
            eval_mode=True,
            save_results=False,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return result


def _pdf_ocr_infer_settings() -> dict:
    return {
        "prompt": OCR_PDF_PROMPT,
        "base_size": OCR_PDF_BASE_SIZE,
        "image_size": OCR_PDF_IMAGE_SIZE,
        "crop_mode": OCR_PDF_CROP_MODE,
        "ngram_window": OCR_PDF_NGRAM_WINDOW,
        "image_mode": "base",
    }


def _run_unlimited_ocr_infer_multi(image_paths: List[str], out_dir: str) -> str:
    pdf_settings = _pdf_ocr_infer_settings()
    _configure_torch_for_ocr()
    with torch.inference_mode():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        result = ocr_model.infer_multi(
            ocr_tokenizer,
            prompt=pdf_settings["prompt"],
            image_files=image_paths,
            output_path=out_dir,
            image_size=pdf_settings["image_size"],
            max_length=OCR_MAX_LENGTH,
            no_repeat_ngram_size=OCR_NO_REPEAT_NGRAM_SIZE,
            ngram_window=pdf_settings["ngram_window"],
            eval_mode=True,
            save_results=False,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return result


def _patch_transformers_for_unlimited_ocr() -> None:
    """Remote Unlimited-OCR code still imports symbols removed in transformers 5.x."""
    import transformers
    import transformers.utils.import_utils as import_utils

    shims = {
        "is_torch_fx_available": lambda: True,
    }
    for name, fn in shims.items():
        if not hasattr(import_utils, name):
            setattr(import_utils, name, fn)

    print(f"transformers {transformers.__version__} (Unlimited-OCR compatibility shim applied)")


def unload_ocr_model() -> None:
    global ocr_tokenizer, ocr_model, hf_space_client
    ocr_tokenizer = None
    ocr_model = None
    hf_space_client = None
    _reset_cuda_state()


def _get_hf_space_client():
    global hf_space_client
    if hf_space_client is not None:
        return hf_space_client

    import inspect
    import time

    import httpx
    from gradio_client import Client

    client_kwargs = {
        "verbose": False,
        "httpx_kwargs": {
            "timeout": httpx.Timeout(HF_SPACE_TIMEOUT, connect=120.0),
        },
    }
    if HF_TOKEN:
        param = "token"
        if "token" not in inspect.signature(Client.__init__).parameters:
            param = "hf_token"
        client_kwargs[param] = HF_TOKEN

    last_exc = None
    for attempt in range(1, HF_SPACE_RETRIES + 1):
        try:
            print(f"Connecting to HF Space ({attempt}/{HF_SPACE_RETRIES})...")
            hf_space_client = Client(HF_SPACE_URL, **client_kwargs)
            if HF_TOKEN:
                print("HF Space client authenticated with HF_TOKEN")
            else:
                print("HF Space client without token — set HF_TOKEN in config cell")
            return hf_space_client
        except Exception as exc:
            last_exc = exc
            hf_space_client = None
            if attempt < HF_SPACE_RETRIES:
                wait = 30 * attempt
                print(f"HF Space connection failed: {exc}. Retrying in {wait}s...")
                time.sleep(wait)

    raise RuntimeError(
        f"Could not connect to HF Space after {HF_SPACE_RETRIES} attempts: {last_exc}"
    ) from last_exc


def _hf_space_prompt_text(prompt: Optional[str] = None) -> str:
    return (prompt or OCR_PROMPT).replace("<image>", "").strip()


def _extract_hf_space_stream_text(update) -> Tuple[str, bool]:
    if isinstance(update, dict):
        return str(update.get("text") or ""), bool(update.get("done"))
    if isinstance(update, (list, tuple)):
        for item in reversed(update):
            if isinstance(item, dict) and "text" in item:
                return str(item.get("text") or ""), bool(item.get("done"))
    return str(update or ""), True


def _hf_space_result_text(result) -> str:
    if isinstance(result, dict):
        return str(result.get("text") or "")
    if isinstance(result, str):
        return result
    if hasattr(result, "__iter__"):
        text = ""
        for update in result:
            chunk, done = _extract_hf_space_stream_text(update)
            if chunk:
                text = chunk
            if done:
                break
        return text
    return str(result or "")


def _ocr_page_hf_space(
    image_path: str,
    *,
    mode: str = "base",
    prompt: Optional[str] = None,
) -> str:
    import time
    from gradio_client import handle_file

    prompt_text = _hf_space_prompt_text(prompt)
    predict_kwargs = {
        "image_path": handle_file(image_path),
        "mode": mode,
        "prompt": prompt_text,
        "api_name": "/run_ocr",
    }

    last_exc = None
    for attempt in range(1, HF_SPACE_RETRIES + 1):
        try:
            client = _get_hf_space_client()
            result = client.predict(**predict_kwargs)
            text = _hf_space_result_text(result)

            if not text.strip():
                job = client.submit(**predict_kwargs)
                for update in job:
                    chunk, done = _extract_hf_space_stream_text(update)
                    if chunk:
                        text = chunk
                    if done:
                        break
                if not text.strip():
                    text = _hf_space_result_text(job.result())

            if text.strip():
                return normalize_unlimited_ocr_page(text)

            last_exc = RuntimeError("empty OCR response")
        except Exception as exc:
            last_exc = exc
            global hf_space_client
            hf_space_client = None
            msg = str(exc)
            if "ZeroGPU quota" in msg or "zero gpu quota" in msg.lower():
                raise RuntimeError(
                    "HF Space ZeroGPU quota agotada. Opciones:\n"
                    "  1) Esperar a que se reinicie la cuota (~24h en plan gratuito)\n"
                    "  2) Hugging Face PRO (más cuota ZeroGPU)\n"
                    "  3) Cambiar OCR_BACKEND a 'sglang' o 'local' en la celda de config\n"
                    "  4) Usar el notebook LightOnOCR mientras tanto\n"
                    f"Detalle: {msg}"
                ) from exc
            if attempt < HF_SPACE_RETRIES:
                wait = 30 * attempt
                print(f"OCR attempt {attempt} failed ({exc}). Retrying in {wait}s...")
                time.sleep(wait)

    raise RuntimeError(f"HF Space OCR failed after {HF_SPACE_RETRIES} attempts: {last_exc}") from last_exc


def load_ocr_model() -> None:
    global ocr_tokenizer, ocr_model
    if OCR_BACKEND == "hf_space":
        print(f"HF Space OCR configured ({HF_SPACE_URL}) — connects on first page")
        return
    if OCR_BACKEND == "sglang":
        check_sglang_server()
        return
    if ocr_model is not None:
        return
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Unlimited-OCR local backend requires CUDA. "
            "Set OCR_BACKEND='sglang' and run an SGLang server, or use a CUDA machine."
        )

    from transformers import AutoModel, AutoTokenizer

    _configure_torch_for_ocr()
    _patch_transformers_for_unlimited_ocr()
    _patch_sdpa_for_unlimited_ocr()
    ocr_dtype = _resolve_ocr_dtype()
    ocr_tokenizer = AutoTokenizer.from_pretrained(OCR_MODEL_ID, trust_remote_code=True)
    ocr_model = AutoModel.from_pretrained(
        OCR_MODEL_ID,
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=ocr_dtype,
    ).eval().cuda()
    if ocr_dtype == torch.bfloat16:
        ocr_model = ocr_model.to(dtype=torch.bfloat16)
    elif ocr_dtype == torch.float32:
        ocr_model = ocr_model.float()
    _prepare_ocr_model(ocr_model)
    print(f"Unlimited-OCR loaded on cuda ({OCR_MODEL_ID}, dtype={ocr_dtype})")
    print(
        "Env:",
        f"torch={torch.__version__}",
        f"cuda={torch.version.cuda}",
        f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a'}",
    )
    print(
        "Local OCR is fragile outside torch==2.10.0 + transformers==4.57.1. "
        "Prefer OCR_BACKEND='sglang'."
    )


def check_sglang_server() -> None:
    url = f"{SGLANG_SERVER_URL.rstrip('/')}/v1/models"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        models = [m.get("id") for m in r.json().get("data", [])]
        print(f"SGLang OK at {SGLANG_SERVER_URL} — models: {models}")
        if SGLANG_MODEL_NAME not in models:
            print(f"Warning: {SGLANG_MODEL_NAME!r} not listed; server may still work.")
    except Exception as exc:
        raise RuntimeError(
            f"SGLang server not reachable at {SGLANG_SERVER_URL}. "
            "Start it with the setup cell above, then rerun."
        ) from exc


def _clean_unlimited_ocr_output(text: str) -> str:
    if not text:
        return ""
    out = _UNLIMITED_DET_RE.sub("", text)
    out = _UNLIMITED_REF_RE.sub(lambda m: m.group(1), out)
    out = out.replace("\\coloneqq", ":=").replace("\\eqqcolon", "=:")
    return out.strip()


def _escape_html_cell(value: str) -> str:
    return (
        value.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


def _markdown_table_block_to_html(table_lines: List[str]) -> str:
    rows = []
    for idx, line in enumerate(table_lines):
        if idx == 1 and _MD_TABLE_SEP_RE.match(line):
            continue
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        if cells:
            rows.append(cells)
    if not rows:
        return "\n".join(table_lines)
    html_rows = []
    for row in rows:
        tds = "".join(f"<td>{_escape_html_cell(c)}</td>" for c in row)
        html_rows.append(f"<tr>{tds}</tr>")
    return "<table>" + "".join(html_rows) + "</table>"


def _markdown_tables_to_html(text: str) -> str:
    if "<table" in text.lower():
        return text
    lines = text.splitlines()
    out: List[str] = []
    i = 0
    while i < len(lines):
        if (
            _MD_TABLE_ROW_RE.match(lines[i])
            and i + 1 < len(lines)
            and _MD_TABLE_SEP_RE.match(lines[i + 1])
        ):
            block = []
            while i < len(lines) and _MD_TABLE_ROW_RE.match(lines[i]):
                block.append(lines[i])
                i += 1
            out.append(_markdown_table_block_to_html(block))
        else:
            out.append(lines[i])
            i += 1
    return "\n".join(out)


def normalize_unlimited_ocr_page(text: str) -> str:
    cleaned = _clean_unlimited_ocr_output(text)
    return _markdown_tables_to_html(cleaned)


def _configure_torch_for_gliner() -> None:
    os.environ.setdefault("PYTORCH_JIT", "0")
    os.environ.setdefault("PYTORCH_NVFUSER_DISABLE", "1")
    for name, args in [
        ("_jit_set_profiling_executor", (False,)),
        ("_jit_set_profiling_mode", (False,)),
        ("_jit_override_can_fuse_on_gpu", (False,)),
        ("_jit_override_can_fuse_on_cpu", (False,)),
        ("_jit_set_texpr_fuser_enabled", (False,)),
        ("_jit_set_nvfuser_enabled", (False,)),
    ]:
        fn = getattr(torch._C, name, None)
        if fn is not None:
            try:
                fn(*args)
            except Exception:
                pass


def load_gliner2_model() -> None:
    global gliner2_model
    if gliner2_model is not None:
        return
    from gliner2 import GLiNER2

    _configure_torch_for_gliner()
    if GLINER2_FORCE_CPU:
        map_location = "cpu"
    else:
        map_location = "cuda" if torch.cuda.is_available() else "cpu"

    gliner2_model = GLiNER2.from_pretrained(GLINER2_MODEL_ID, map_location=map_location)
    print(f"GLiNER2 loaded on {map_location}")


def _reset_cuda_state() -> None:
    if not torch.cuda.is_available():
        return
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


def _prepare_page_image_for_ocr(img: Image.Image) -> Image.Image:
    img = img.convert("RGB") if img.mode != "RGB" else img
    w, h = img.size
    longest = max(w, h)
    if longest > OCR_MAX_PAGE_LONGEST:
        ratio = OCR_MAX_PAGE_LONGEST / longest
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    return img


def render_pdf_page(pdf_doc: fitz.Document, page_idx: int, dpi: int = OCR_RENDER_DPI) -> Image.Image:
    page = pdf_doc[page_idx]
    pix = page.get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    return _prepare_page_image_for_ocr(img)


def _ocr_page_local(image_path: str, **infer_overrides) -> str:
    try:
        with tempfile.TemporaryDirectory(prefix="unlimited_ocr_") as out_dir:
            raw = _run_unlimited_ocr_infer(image_path, out_dir, **infer_overrides)
        return normalize_unlimited_ocr_page(str(raw or ""))
    except RuntimeError as exc:
        _reset_cuda_state()
        msg = str(exc).lower()
        if "cuda" in msg or "cublas" in msg or "gather" in msg:
            raise RuntimeError(
                "Unlimited-OCR local CUDA failure. Restart the kernel, then either "
                "lower OCR_MAX_PAGE_LONGEST / OCR_RENDER_DPI, or switch to "
                "OCR_BACKEND='sglang'."
            ) from exc
        raise


def _encode_image_base64(image_path: str) -> dict:
    ext = os.path.splitext(image_path)[1].lower()
    mime = "image/jpeg" if ext in (".jpg", ".jpeg") else f"image/{ext.lstrip('.')}"
    with open(image_path, "rb") as f:
        data = base64.b64encode(f.read()).decode("utf-8")
    return {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}}


def _ocr_page_sglang(
    image_path: str,
    *,
    image_mode: Optional[str] = None,
    ngram_window: Optional[int] = None,
    prompt: Optional[str] = None,
) -> str:
    pdf_settings = _pdf_ocr_infer_settings()
    payload = {
        "model": SGLANG_MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (prompt or OCR_PROMPT).replace("<image>", ""),
                    },
                    _encode_image_base64(image_path),
                ],
            }
        ],
        "temperature": 0,
        "skip_special_tokens": False,
        "images_config": {"image_mode": image_mode or ("gundam" if OCR_CROP_MODE else "base")},
        "custom_params": {
            "ngram_size": OCR_NO_REPEAT_NGRAM_SIZE,
            "window_size": ngram_window or OCR_NGRAM_WINDOW,
        },
        "stream": False,
    }
    r = requests.post(
        f"{SGLANG_SERVER_URL.rstrip('/')}/v1/chat/completions",
        json=payload,
        timeout=1200,
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]
    return normalize_unlimited_ocr_page(str(content or ""))


def _ocr_pdf_sglang_multi(image_paths: List[str]) -> str:
    pdf_settings = _pdf_ocr_infer_settings()
    content = [{"type": "text", "text": pdf_settings["prompt"].replace("<image>", "")}]
    content.extend(_encode_image_base64(path) for path in image_paths)
    payload = {
        "model": SGLANG_MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "skip_special_tokens": False,
        "images_config": {"image_mode": pdf_settings["image_mode"]},
        "custom_params": {
            "ngram_size": OCR_NO_REPEAT_NGRAM_SIZE,
            "window_size": pdf_settings["ngram_window"],
        },
        "stream": False,
    }
    r = requests.post(
        f"{SGLANG_SERVER_URL.rstrip('/')}/v1/chat/completions",
        json=payload,
        timeout=3600,
    )
    r.raise_for_status()
    return str(r.json()["choices"][0]["message"]["content"] or "")


def _split_multi_page_ocr_output(text: str, num_pages: int) -> List[str]:
    text = str(text or "").strip()
    if not text:
        return [""] * num_pages
    if num_pages == 1:
        return [text]

    chunks = [c.strip() for c in re.split(r"\f+", text) if c.strip()]
    if len(chunks) == num_pages:
        return chunks

    chunks = [c.strip() for c in re.split(r"\n-{3,}\n+", text) if c.strip()]
    if len(chunks) == num_pages:
        return chunks

    page_markers = list(re.finditer(r"(?im)^(?:#+\s*)?page\s+\d+\b", text))
    if len(page_markers) >= num_pages:
        starts = [m.start() for m in page_markers[:num_pages]]
        starts.append(len(text))
        return [text[starts[i]:starts[i + 1]].strip() for i in range(num_pages)]

    raise RuntimeError(
        f"Could not split multi-page OCR into {num_pages} pages (got {len(chunks)} chunks)"
    )


def _save_temp_page_images(images: List[Image.Image]) -> List[str]:
    paths: List[str] = []
    for img in images:
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        img.save(tmp, format="PNG")
        tmp.close()
        paths.append(tmp.name)
    return paths


def _cleanup_temp_files(paths: List[str]) -> None:
    for path in paths:
        try:
            os.unlink(path)
        except OSError:
            pass


def _ocr_pdf_per_page_base(pdf_doc: fitz.Document) -> List[str]:
    pdf_settings = _pdf_ocr_infer_settings()
    infer_overrides = {
        "prompt": pdf_settings["prompt"],
        "base_size": pdf_settings["base_size"],
        "image_size": pdf_settings["image_size"],
        "crop_mode": pdf_settings["crop_mode"],
        "ngram_window": pdf_settings["ngram_window"],
    }
    sglang_kwargs = {
        "prompt": pdf_settings["prompt"],
        "image_mode": pdf_settings["image_mode"],
        "ngram_window": pdf_settings["ngram_window"],
    }
    hf_space_mode = HF_SPACE_PDF_MODE if OCR_BACKEND == "hf_space" else pdf_settings["image_mode"]
    pages: List[str] = []
    for page_idx in range(len(pdf_doc)):
        img = render_pdf_page(pdf_doc, page_idx)
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        img.save(tmp, format="PNG")
        tmp.close()
        try:
            if OCR_BACKEND == "hf_space":
                pages.append(
                    _ocr_page_hf_space(
                        tmp.name,
                        mode=hf_space_mode,
                        prompt=pdf_settings["prompt"],
                    )
                )
            elif OCR_BACKEND == "sglang":
                pages.append(_ocr_page_sglang(tmp.name, **sglang_kwargs))
            else:
                pages.append(_ocr_page_local(tmp.name, **infer_overrides))
        finally:
            os.unlink(tmp.name)
        if OCR_BACKEND == "local" and torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pages


def _ocr_pdf_multi_base(pdf_doc: fitz.Document) -> List[str]:
    images = [render_pdf_page(pdf_doc, page_idx) for page_idx in range(len(pdf_doc))]
    image_paths = _save_temp_page_images(images)
    try:
        if OCR_BACKEND == "sglang":
            raw = _ocr_pdf_sglang_multi(image_paths)
        else:
            with tempfile.TemporaryDirectory(prefix="unlimited_ocr_multi_") as out_dir:
                raw = _run_unlimited_ocr_infer_multi(image_paths, out_dir)
        chunks = _split_multi_page_ocr_output(str(raw or ""), len(images))
        return [normalize_unlimited_ocr_page(chunk) for chunk in chunks]
    finally:
        _cleanup_temp_files(image_paths)


def ocr_page(img: Image.Image) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
    img.save(tmp, format="PNG")
    tmp.close()
    try:
        if OCR_BACKEND == "hf_space":
            mode = "gundam" if OCR_CROP_MODE else "base"
            return _ocr_page_hf_space(tmp.name, mode=mode)
        if OCR_BACKEND == "sglang":
            return _ocr_page_sglang(tmp.name)
        return _ocr_page_local(tmp.name)
    finally:
        os.unlink(tmp.name)


def ocr_pdf_pages(pdf_path: Path, cache_dir: Path, *, force_ocr: bool = False) -> List[str]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"{pdf_path.stem}_pages.json"
    if cache_file.exists() and not force_ocr:
        with cache_file.open(encoding="utf-8") as f:
            payload = json.load(f)
            return payload["pages"]

    pdf_doc = fitz.open(str(pdf_path))
    pages: List[str] = []
    try:
        if OCR_PDF_STRATEGY == "multi_base" and OCR_BACKEND in {"local", "sglang"}:
            try:
                pages = _ocr_pdf_multi_base(pdf_doc)
            except Exception as exc:
                print(f"multi_base failed ({exc}); falling back to per_page_base")
                pages = _ocr_pdf_per_page_base(pdf_doc)
        else:
            pages = _ocr_pdf_per_page_base(pdf_doc)
    finally:
        pdf_doc.close()

    with cache_file.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "file_name": str(pdf_path),
                "ocr_backend": OCR_BACKEND,
                "ocr_model": OCR_MODEL_ID,
                "ocr_pdf_strategy": OCR_PDF_STRATEGY,
                "ocr_pdf_mode": "base",
                "pages": pages,
            },
            f,
            ensure_ascii=False,
        )
    return pages


def normalize_table_number(raw: str) -> str:
    return raw.strip().upper().replace(" ", "")


def _collapse_ws(text: str) -> str:
    return re.sub(r"[ \t]+", " ", text).strip()


def _repair_hyphen_breaks(text: str) -> str:
    return re.sub(r"(\w)-\s*\n\s*([a-z][\w'-]*)", r"\1\2", text)


def _split_paragraphs(text: str) -> List[str]:
    text = _repair_hyphen_breaks(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) > 1:
        return [_collapse_ws(p) for p in paras]
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if len(lines) <= 1:
        return [_collapse_ws(text)] if text.strip() else []
    chunks, buf = [], []
    for line in lines:
        buf.append(line)
        if line.endswith((".", "!", "?", '."')):
            chunks.append(" ".join(buf))
            buf = []
    if buf:
        chunks.append(" ".join(buf))
    if len(chunks) > 1:
        return [_collapse_ws(p) for p in chunks if p.strip()]
    return [_collapse_ws(text)] if text.strip() else []


def _table_nums_in_para(para: str) -> set:
    nums = set()
    for m in TABLE_REF_RE.finditer(para):
        body = _REF_KEYWORD_RE.sub("", m.group(0))
        for g in _REF_NUM_RE.findall(body):
            nums.add(normalize_table_number(g))
    return nums


def find_page_captions(page_text: str) -> List[dict]:
    captions = []
    for m in CAPTION_RE.finditer(page_text):
        start = m.start()
        para = re.split(r"\n\s*\n", page_text[start:], maxsplit=1)[0]
        caption = _collapse_ws(para)
        if caption:
            captions.append({
                "table_num": normalize_table_number(m.group(1)),
                "caption": caption,
                "start": start,
                "end": start + len(para),
            })
    return captions


def pair_captions_to_tables(page_text: str) -> List[dict]:
    blocks = list(TABLE_BLOCK_RE.finditer(page_text))
    captions = find_page_captions(page_text)
    if not blocks:
        return []
    results = []
    if len(captions) == len(blocks):
        for bm, cap in zip(blocks, sorted(captions, key=lambda c: c["start"])):
            results.append({"match": bm, "caption": cap["caption"], "table_num": cap["table_num"]})
        return results
    used = set()
    for bm in blocks:
        t_start, t_end = bm.start(), bm.end()
        best_idx, best_dist = None, float("inf")
        for ci, cap in enumerate(captions):
            if ci in used:
                continue
            if cap["end"] <= t_start:
                dist = t_start - cap["end"]
            elif cap["start"] >= t_end:
                dist = cap["start"] - t_end
            else:
                dist = 0
            if dist < best_dist:
                best_dist, best_idx = dist, ci
        if best_idx is not None:
            used.add(best_idx)
            cap = captions[best_idx]
            results.append({"match": bm, "caption": cap["caption"], "table_num": cap["table_num"]})
        else:
            results.append({"match": bm, "caption": "", "table_num": None})
    return results


def page_paragraphs_without_tables(page_text: str) -> List[str]:
    return _split_paragraphs(TABLE_BLOCK_RE.sub(" ", page_text))


def _is_safe_continuation(para: str) -> bool:
    t = para.strip()
    return bool(t) and not CAPTION_RE.match(t) and not re.match(r"^Table\b", t, re.IGNORECASE)


def _ends_complete_sentence(text: str) -> bool:
    return bool(_SENTENCE_END_RE.search(text.rstrip()))


def _continuation_paragraphs(page_idx: int, para_idx: int, paras: List[str], pages: List[str]):
    for k in range(para_idx + 1, len(paras)):
        yield paras[k]
    if page_idx < len(pages):
        for p in page_paragraphs_without_tables(pages[page_idx]):
            yield p


def _extract_hyphen_continuation_prefix(para: str) -> Optional[str]:
    t = para.lstrip()
    if not t or not t[0].islower() or CAPTION_RE.match(t) or re.match(r"^Table\b", t, re.IGNORECASE):
        return None
    m = _HYPHEN_CONTINUATION_PREFIX_RE.match(t)
    return (m.group(1) + (m.group(2) or "")) if m else None


def _extend_mention_text(para: str, page_idx: int, para_idx: int, paras: List[str], pages: List[str]) -> str:
    out, extensions = para.strip(), 0
    for nxt in _continuation_paragraphs(page_idx, para_idx, paras, pages):
        if extensions >= MAX_MENTION_EXTENSIONS:
            break
        nxt_text = nxt.strip()
        if not nxt_text:
            continue
        if out.rstrip().endswith("-"):
            merged_base = out.rstrip()[:-1]
            if _is_safe_continuation(nxt_text):
                out = _collapse_ws(merged_base + nxt_text)
                extensions += 1
                continue
            prefix = _extract_hyphen_continuation_prefix(nxt_text)
            if prefix:
                out = _collapse_ws(merged_base + prefix)
                break
            continue
        if _ends_complete_sentence(out) or out.rstrip().endswith(":"):
            break
        if not _is_safe_continuation(nxt_text):
            continue
        if extensions == 0 and not nxt_text[0].islower():
            break
        out = _collapse_ws(out + " " + nxt_text)
        extensions += 1
    return out


def _is_usable_mention(text: str) -> bool:
    t = text.strip()
    if len(t) < MIN_MENTION_CHARS or re.fullmatch(r"-+", t):
        return False
    if _HEADING_ONLY_RE.match(t) and len(t) < 80:
        return False
    if t.startswith("$$") or t.startswith("\\["):
        return False
    url_hits = len(_FOOTNOTE_URL_RE.findall(t))
    if url_hits >= 2:
        return False
    if url_hits >= 1 and len(t) < 200:
        return False
    return True


def find_mentions(pages: List[str]) -> Dict[str, List[dict]]:
    mentions, seen = {}, {}
    for page_idx, page_text in enumerate(pages, start=1):
        paras = page_paragraphs_without_tables(page_text)
        for i, para in enumerate(paras):
            if CAPTION_RE.match(para):
                continue
            nums = _table_nums_in_para(para)
            if not nums:
                continue
            lo, hi = max(0, i - ADJACENT_PARA_WINDOW), min(len(paras), i + ADJACENT_PARA_WINDOW + 1)
            for j in range(lo, hi):
                candidate = paras[j]
                if CAPTION_RE.match(candidate):
                    continue
                if j != i:
                    adj_nums = _table_nums_in_para(candidate)
                    if adj_nums and adj_nums.isdisjoint(nums):
                        continue
                text = _extend_mention_text(candidate, page_idx, j, paras, pages)
                if not _is_usable_mention(text):
                    continue
                for n in nums:
                    key = (page_idx, text)
                    seen.setdefault(n, set())
                    if key in seen[n]:
                        continue
                    seen[n].add(key)
                    mentions.setdefault(n, []).append({"page": page_idx, "text": text})
    for n in mentions:
        mentions[n].sort(key=lambda m: (m["page"], m["text"]))
    return mentions


def header_and_rows_from_html(html: str) -> Tuple[List[str], List[str]]:
    soup = BeautifulSoup(html, "html.parser")
    thead, tbody = soup.find("thead"), soup.find("tbody")
    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None
    header_lines, body_lines = [], []
    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)
    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for tr in trs:
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        if all(c.name == "th" for c in cells) and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)
    if not header_lines and body_lines:
        header_lines = [body_lines[0]]
        body_lines = body_lines[1:]
    return header_lines, body_lines


def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = _strip_accents(str(text).strip().lower())
    return re.sub(r"\s+", " ", s)


def normalize_dataset(text: object) -> str:
    return normalize_text(text).replace(" ", "").replace("_", "").replace("-", "")


def _strip_trailing_plural(s: str) -> str:
    if not s or "@" in s or not s.isalpha():
        return s
    if len(s) > 3 and s.endswith("s") and not s.endswith("ss"):
        return s[:-1]
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    return _strip_trailing_plural(re.sub(r"[^a-z0-9@]+", "", s))


def metric_dedup_key(raw: str) -> str:
    nm = normalize_metric(raw)
    if not nm:
        return ""
    m = re.match(r"^h@(\d+)$", nm)
    if m:
        return f"hits@{m.group(1)}"
    return nm


def _is_incomplete_metric_key(key: str) -> bool:
    if not key or "@" not in key:
        return False
    suffix = key.split("@", 1)[1]
    return not suffix or not any(ch.isdigit() for ch in suffix)


def _pick_display(candidates: List[str]) -> str:
    return max(candidates, key=lambda s: (len(s), any(c.isupper() for c in s)))


def _dedup_by_key(items: List[str], key_fn) -> List[str]:
    buckets: Dict[str, List[str]] = {}
    for value in items:
        key = key_fn(value)
        if not key:
            continue
        buckets.setdefault(key, []).append(value)
    return sorted(_pick_display(vals) for vals in buckets.values())


def _clean_entity(value: str) -> str:
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    s = re.sub(r"\\(?:textbf|textit|text|mathbf|mathrm|mathit)\{([^{}]*)\}", r"\1", s)
    for _ in range(3):
        s = re.sub(r"[_^]\{([^{}]*)\}", r"\1", s)
    s = s.replace("{", "").replace("}", "")
    return re.sub(r"\s+", " ", s).strip(" .,:;-")


def _get_latex2text():
    global _latex2text
    if _latex2text is None:
        from pylatexenc.latex2text import LatexNodes2Text
        _latex2text = LatexNodes2Text()
    return _latex2text


def _convert_latex_fragment(fragment: str) -> str:
    try:
        return _get_latex2text().latex_to_text(fragment)
    except Exception:
        return fragment


def latex_to_plain(text: str) -> str:
    if not text or not str(text).strip():
        return text
    def _repl(match: re.Match) -> str:
        fragment = next(g for g in match.groups() if g is not None)
        return _convert_latex_fragment(fragment)
    return _LATEX_MATH_RE.sub(_repl, str(text))


def _format_mentions_for_export(mentions: List[dict]) -> List[dict]:
    return [{"page": m["page"], "text": latex_to_plain(m["text"])} for m in mentions]


print("Standalone pipeline loaded (Unlimited-OCR)")
_configure_torch_for_ocr()

In [ ]:
def _extract_json_object(text: str) -> Dict:
    text = text.strip()
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return {}
    blob = m.group(0)
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        return {}


_ORPHAN_PREFIX_RE = re.compile(
    r"^(?:reported|shown|presented|compared|listed|denoted|obtained|computed|evaluated)\b",
    re.IGNORECASE,
)


def _looks_orphaned_fragment(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    first = t[0]
    if first.islower() and _ORPHAN_PREFIX_RE.match(t):
        return True
    if first.islower() and re.search(r"\bin\s+the\s+table\s+\d+\b", t, re.IGNORECASE):
        return True
    return False


def _normalize_mention_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip()).lower()


def _dedup_mentions(mentions: List[dict]) -> List[dict]:
    seen = set()
    out = []
    for m in mentions:
        page = m.get("page")
        text = str(m.get("text", "")).strip()
        if not isinstance(page, int) or not text:
            continue
        key = (page, _normalize_mention_text(text))
        if key in seen:
            continue
        seen.add(key)
        out.append({"page": page, "text": text})
    return out


def _mentions_explicitly_for_table(mentions: List[dict], table_num: Optional[str]) -> List[dict]:
    if not table_num:
        return _dedup_mentions(mentions)
    pat = re.compile(rf"\b(?:table|tab\.?)\s*{re.escape(table_num)}\b", re.IGNORECASE)
    out = []
    for m in mentions:
        text = str(m.get("text", "")).strip()
        page = m.get("page")
        if isinstance(page, int) and text and pat.search(text):
            out.append({"page": page, "text": text})
    return _dedup_mentions(out)


def _ollama_refine_mentions(table_label: str, caption: str, mentions: List[dict]) -> List[dict]:
    if not mentions:
        return mentions

    table_num_match = re.search(r"\b(\d+(?:\.\d+)?)\b", table_label or "")
    table_num = table_num_match.group(1) if table_num_match else None

    instruction = {
        "task": "filter_mentions_for_one_table_keep_full_text",
        "strict_rules": [
            "Keep only mentions clearly about this specific table.",
            "Prefer mentions that explicitly cite the target table number.",
            "Drop mentions that are mainly about other tables.",
            "Do not shorten: keep full original mention text whenever possible.",
            "Do not return sentence fragments that start mid-thought (e.g., 'reported in the Table ...').",
            "Do not invent content.",
            "Preserve page numbers from input mentions.",
            "Output JSON only with key 'mentions'.",
        ],
        "target_table_label": table_label,
        "target_table_number": table_num,
        "target_caption": caption,
        "input_mentions": mentions,
        "output_schema": {"mentions": [{"page": "int", "text": "str"}]},
    }

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": json.dumps(instruction, ensure_ascii=False),
        "stream": False,
        "options": {"temperature": 0},
    }

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=OLLAMA_TIMEOUT)
        r.raise_for_status()
        response_text = r.json().get("response", "")
        data = _extract_json_object(response_text)
        if not isinstance(data, dict) or "mentions" not in data:
            return mentions  # parse failure fallback

        raw_mentions = data.get("mentions", [])
        clean = []
        for m in raw_mentions:
            if not isinstance(m, dict):
                continue
            page = m.get("page")
            text = str(m.get("text", "")).strip()
            if isinstance(page, int) and text and not _looks_orphaned_fragment(text):
                clean.append({"page": page, "text": text})

        clean = _dedup_mentions(clean)
        if clean:
            return clean

        # If LLM over-filters to empty, fallback to explicit Table N mentions.
        explicit = _mentions_explicitly_for_table(mentions, table_num)
        if explicit:
            return explicit
        return _dedup_mentions(mentions)
    except Exception:
        return _dedup_mentions(mentions)

In [ ]:
# (SGLang health check moved to the run cell below)

In [ ]:
def _extract_entities_raw_gliner(text: str) -> Dict[str, Tuple[str, float]]:
    best: Dict[str, Tuple[str, float]] = {}
    if not text or not text.strip():
        return best
    result = gliner2_model.extract_entities(
        text[:GLINER2_MAX_CHARS],
        GLINER2_LABEL_DESCRIPTIONS,
        include_confidence=True,
    )
    for label, items in (result or {}).get("entities", {}).items():
        if label not in GLINER2_LABELS:
            continue
        for item in items or []:
            if isinstance(item, dict):
                raw, score = str(item.get("text", "")), float(item.get("confidence", 1.0) or 1.0)
            else:
                raw, score = str(item), 1.0
            value = _clean_entity(raw)
            if score < GLINER2_MIN_SCORE or len(value) < 2:
                continue
            if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
                continue
            prev = best.get(value)
            if prev is None or score > prev[1]:
                best[value] = (label, score)
    return best


def _merge_best(target: Dict[str, Tuple[str, float]], other: Dict[str, Tuple[str, float]]) -> None:
    for value, (label, score) in other.items():
        prev = target.get(value)
        if prev is None or score > prev[1]:
            target[value] = (label, score)


def _keys_from_best(best: Dict[str, Tuple[str, float]], label: str, key_fn) -> set:
    keys = set()
    for value, (lbl, _) in best.items():
        if lbl != label:
            continue
        k = key_fn(value)
        if k:
            keys.add(k)
    return keys


def extract_table_entities_gliner(caption: str, html: str) -> Dict[str, List[str]]:
    header_lines, body_lines = header_and_rows_from_html(html)
    header_context = "\n".join(header_lines)

    table_best: Dict[str, Tuple[str, float]] = {}
    if header_context:
        _merge_best(table_best, _extract_entities_raw_gliner(header_context))

    for row_text in body_lines:
        prompt = f"Table column headers: {header_context}\nRow: {row_text}" if header_context else row_text
        _merge_best(table_best, _extract_entities_raw_gliner(prompt))

    if not table_best:
        _merge_best(table_best, _extract_entities_raw_gliner("\n".join(header_lines + body_lines)))

    caption_only_ds: set = set()
    caption_only_mt: set = set()
    if caption:
        cap_best = _extract_entities_raw_gliner(caption)
        cap_ds = _keys_from_best(cap_best, "dataset", normalize_dataset)
        cap_mt = _keys_from_best(cap_best, "metric", metric_dedup_key)
        table_ds = _keys_from_best(table_best, "dataset", normalize_dataset)
        table_mt = _keys_from_best(table_best, "metric", metric_dedup_key)
        caption_only_ds = cap_ds - table_ds
        caption_only_mt = cap_mt - table_mt

    filtered_datasets: List[str] = []
    filtered_metrics: List[str] = []
    for value, (label, _) in table_best.items():
        if label == "model":
            continue
        if label == "dataset":
            k = normalize_dataset(value)
            if k and k not in caption_only_ds:
                filtered_datasets.append(value)
        elif label == "metric":
            k = metric_dedup_key(value)
            if k and not _is_incomplete_metric_key(k) and k not in caption_only_mt:
                filtered_metrics.append(value)

    return {
        "dataset": _dedup_by_key(filtered_datasets, normalize_dataset),
        "metric": _dedup_by_key(filtered_metrics, metric_dedup_key),
    }


def extract_context_for_pdf(pdf_path: Path, *, ocr_cache_dir: Path, force_ocr: bool = False) -> dict:
    pages = ocr_pdf_pages(pdf_path, ocr_cache_dir, force_ocr=force_ocr)
    mentions_by_num = find_mentions(pages)

    tables = []
    for page_idx, page_text in enumerate(pages, start=1):
        paired = pair_captions_to_tables(page_text)
        for t_i, item in enumerate(paired, start=1):
            html = item["match"].group(0)
            caption_raw = item["caption"] or ""
            table_num = item["table_num"]
            table_label = f"Table {table_num}" if table_num else None
            mentions_raw = mentions_by_num.get(table_num, []) if table_num else []
            if USE_OLLAMA_FOR_CONTEXT:
                mentions_raw = _ollama_refine_mentions(table_label or "Table", caption_raw, mentions_raw)

            ents = extract_table_entities_gliner(caption_raw, html)

            tables.append(
                {
                    "table_id": f"{pdf_path.stem}_p{page_idx}_t{t_i}",
                    "table_label": table_label,
                    "page": page_idx,
                    "caption": latex_to_plain(caption_raw),
                    "mentions": _format_mentions_for_export(mentions_raw),
                    "datasets": ents["dataset"],
                    "metrics": ents["metric"],
                }
            )

    return {
        "paper": pdf_path.stem,
        "source_pdf": str(pdf_path.resolve()),
        "num_tables": len(tables),
        "tables": tables,
    }


def collect_pdf_paths(input_path: Path) -> List[Path]:
    input_path = input_path.resolve()
    if input_path.is_file():
        if input_path.suffix.lower() != ".pdf":
            raise ValueError(f"Not a PDF file: {input_path}")
        return [input_path]
    if input_path.is_dir():
        pdfs = sorted(input_path.glob("*.pdf"))
        if not pdfs:
            raise FileNotFoundError(f"No PDF files in {input_path}")
        return pdfs
    raise FileNotFoundError(f"Input path does not exist: {input_path}")


def run_extract(
    input_path: Path,
    output_dir: Path,
    *,
    ocr_cache_dir: Optional[Path] = None,
    force_ocr: bool = False,
    skip_existing: bool = False,
) -> List[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = ocr_cache_dir or (output_dir / "ocr_cache")

    load_ocr_model()
    load_gliner2_model()

    pdf_paths = collect_pdf_paths(input_path)
    written: List[Path] = []

    for i, pdf_path in enumerate(pdf_paths, start=1):
        out_file = output_dir / f"{pdf_path.stem}.json"
        if skip_existing and out_file.exists():
            written.append(out_file)
            continue

        doc = extract_context_for_pdf(
            pdf_path,
            ocr_cache_dir=cache_dir,
            force_ocr=force_ocr,
        )
        with out_file.open("w", encoding="utf-8") as f:
            json.dump(doc, f, indent=2, ensure_ascii=False)
        written.append(out_file)

    return written


print("Hybrid mode loaded: Unlimited-OCR + GLiNER entities + Ollama context")

In [ ]:
# Quick Ollama health check
health_payload = {
    "model": OLLAMA_MODEL,
    "prompt": "Reply with JSON: {\"ok\": true}",
    "stream": False,
    "options": {"temperature": 0},
}
resp = requests.post(OLLAMA_URL, json=health_payload, timeout=OLLAMA_TIMEOUT)
print("HTTP", resp.status_code)
print(resp.json().get("response", "")[:200])

In [ ]:
# Pre-flight checks
if OCR_BACKEND == "sglang":
    check_sglang_server()
elif OCR_BACKEND == "hf_space":
    print(f"HF Space OCR — will connect on first page ({HF_SPACE_URL})")
    if not HF_TOKEN:
        print("Tip: set HF_TOKEN in the config cell for more ZeroGPU quota")
else:
    print("OCR_BACKEND is local — ensure torch==2.10.0 + transformers==4.57.1")

# Run extraction (same input/output behavior as script)
written = run_extract(
    INPUT_PATH,
    OUTPUT_DIR,
    ocr_cache_dir=OCR_CACHE_DIR,
    force_ocr=FORCE_OCR,
    skip_existing=SKIP_EXISTING,
)
print(f"Done. {len(written)} JSON file(s) written to {OUTPUT_DIR.resolve()}")
for p in written:
    print(" -", p)

In [ ]:
# Preview one output file
files = sorted(OUTPUT_DIR.glob("*.json"))
if files:
    sample = files[0]
    data = json.loads(sample.read_text(encoding="utf-8"))
    print("Sample:", sample.name)
    print("paper:", data.get("paper"))
    print("num_tables:", data.get("num_tables"))
    if data.get("tables"):
        t0 = data["tables"][0]
        print("table_id:", t0.get("table_id"))
        print("caption:", t0.get("caption", "")[:180])
        print("mentions:", len(t0.get("mentions", [])))
        print("datasets:", t0.get("datasets", []))
        print("metrics:", t0.get("metrics", []))
else:
    print("No JSON outputs found in", OUTPUT_DIR)